In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
# Cathing the parameter

init_load_flag = int(dbutils.widgets.get("init_load_flag"))

### Reading data from the source

In [0]:

df = spark.sql("select * from databricks_cata.silver.customers_silver")

### Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])
df.limit(10).display()

### Surrogate Keys - All the values

In [0]:
# CREATE AUTO INCREMENT

df = df.withColumn("DimCustomerKey",monotonically_increasing_id()+lit(1))
df.limit(10).display()

### Full Load and Incremental

In [0]:
if init_load_flag == 0: # 0 = Incremental / 1= Full Load
  
  df_old = spark.sql('''Select DimCustomerKey, customer_id, create_date, update_date from databricks_cata.gold.DimCustomer''')
  
else:
    df_old = spark.sql('''  Select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date 
                            from databricks_cata.silver.customers_silver where 1 = 0''')


### Renaming

In [0]:
df_old = df_old.withColumnRenamed("DimCustomerKey","DimCustomerKey_old")\
               .withColumnRenamed("customer_id","customer_id_old")\
               .withColumnRenamed("create_date","create_date_old")\
               .withColumnRenamed("update_date","update_date_old")
df_old.limit(10).display()

### Applying join with the Old Records

In [0]:
df_join = df.join(df_old,df['customer_id'] == df_old['customer_id_old'],'left')

In [0]:
df_join.limit(5).display()

### Separating New vs Old Records

In [0]:
df_new = df_join.filter(df_join['DimCustomerKey_old'].isNull())
df_new.limit(5).display()

In [0]:
df_old = df_join.filter(df_join['DimCustomerKey_old'].isNotNull())
df_old.limit(5).display()

### Preparing df_old

In [0]:
# DROPPING ALL UNNECESSARY COLUMNS
df_old = df_old.drop('DimCustomerKey_old','customer_id_old','update_date_old')

# Renaming column
#df_old = df_old.withColumnRenamed("DimCustomerKey_old","DimCustomerKey")
df_old = df_old.withColumnRenamed("create_date_old","create_date")
df_old = df_old.withColumn("create_date",to_timestamp(col("create_date")))

               
# Recreating "update_date" with TimeStamp
df_old = df_old.withColumn("update_date",current_timestamp())
df_old.limit(5).display()

In [0]:
# DROPPING ALL UNNECESSARY COLUMNS
df_new = df_new.drop('DimCustomerKey_old','customer_id_old','update_date_old','create_date_old')

# Renaming column
df_new = df_new.withColumn("update_date",current_timestamp())
df_new = df_new.withColumn("create_date",current_timestamp())

df_new.limit(5).display()               


### Surrogate Keys - All the values

In [0]:
df_new = df_new.withColumn("DimCustomerKey", monotonically_increasing_id() + lit(1))
df_new.limit(5).display()


### Getting Max Surrogate Key from dimension

In [0]:
if init_load_flag == 1: # 0 = Incremental / 1= Full Load
  max_surrogate_key = 0 
else:
    df_max_surr = spark.sql('''Select max(DimCustomerKey) as max_surrofate_key from databricks_cata.gold.DimCustomer''')

    # Converting df_maxsur to max_surrogate_key
    max_surrogate_key = df_max_surr.collect()[0]['max_surrogate_key']
  

In [0]:
df_new = df_new.withColumn("DimCustomerKey", lit(max_surrogate_key) + col("DimCustomerKey"))

###  Union df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old)

In [0]:
df_final.limit(5).display()

### SCD Type 1

In [0]:
from delta.tables import DeltaTable

In [0]:
if (spark.catalog.tableExists("databricks_cata.gold.Dimcustomers")):

    dlt_obj = DeltaTable.forPath(spark,"abfss://gold@databricksuk2025.dfs.core.windows.net/DimCustomers")

    dlt_obj.alias("trg").merge(df_final.alias("src"),"trg.DimCustomerKey = src.DimCustomerKey")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

else:
    df_final.write.mode("overwrite")\
        .format("delta")\
        .option("path", "abfss://gold@databricksuk2025.dfs.core.windows.net/DimCustomers")\
        .saveAsTable("databricks_cata.gold.DimCustomers")
